# 02. 토지피복 비율 만들기
**작성자: B** (D도 지도 격자에 같은 함수 사용)

결과물: `data_processed/features.csv`

In [ ]:
# ── 공통 설정 (모든 노트북 맨 위에 그대로 두기) ──
from google.colab import drive
drive.mount('/content/drive')

BASE = '/content/drive/MyDrive/AICEE_project'   # 모두 같은 경로 (공유 폴더를 '내 드라이브에 바로가기 추가' 해야 함)
DATA = f'{BASE}/data_processed'
RESULTS = f'{BASE}/results'

import sys
sys.path.append(f'{BASE}/shared')
RANDOM_STATE = 42

USE_DUMMY = True   # ← 진짜 데이터 나오면 False로 바꾸기

> **6주차 누수 점검**: 입력 변수는 '예측하는 순간에 알 수 있는 값'만. 토지피복·한강 거리는 OK. 같은 시각 다른 센서의 기온 같은 건 넣지 않기.

In [ ]:
!pip -q install rasterio

## 1) WorldCover 지도
서울이 들어 있는 타일(`...N36E126_Map.tif` 예상, 받기 전에 확인)을 `data_raw/worldcover/`에 넣기

In [ ]:
import glob, rasterio
tif = glob.glob(f'{BASE}/data_raw/worldcover/*.tif')[0]
with rasterio.open(tif) as src:
    print(src.crs, src.bounds, src.res)

## 2) 먼저 센서 10개로 테스트 → 숫자가 그럴듯한지 확인 (공원 옆 센서는 녹지↑?)

In [ ]:
import pandas as pd
from landcover import add_landcover_columns
sensors = pd.read_csv(f'{DATA}/labels_dummy.csv' if USE_DUMMY else f'{DATA}/labels.csv')[['sensor_id', 'gu', 'lat', 'lon']]
add_landcover_columns(sensors.head(10), tif, radius_m=300)

## 3) 전체 센서 계산 + 한강까지 거리 (TODO)

In [ ]:
feats = add_landcover_columns(sensors, tif, radius_m=300)
# TODO: 한강까지 거리(dist_han_m) 계산 방법 팀에서 정하기
feats['dist_han_m'] = float('nan')
feats.to_csv(f'{DATA}/features.csv', index=False)
feats.describe()

## 4) 변수끼리 상관관계 (2주차 EDA)
녹지+불투수면+물 ≈ 100%라서 서로 강하게 얽혀 있을 것 → 모델 2에서 VIF로 확인

In [ ]:
feats[['green_300', 'impervious_300', 'water_300', 'dist_han_m']].corr().round(2)